# Muon: ортогонализация градиентов и спектральная оптимизация

**Курс «Методы оптимизации», МФТИ**

1. **Newton-Schulz** — быстрое приближение полярного фактора $UV^\top$
2. **Спектр обновлений** — SGD перекашивает, Muon выравнивает
3. **Инвариантность к числу обусловленности** — Muon vs GD vs Adam
4. **Предельный цикл Adam** — осцилляция на $f(w) = w^2$
5. **Shampoo(1 grad) ≡ Muon** — элегантное тождество
6. **MLP на Fashion-MNIST** — Muon vs Adam vs SGD

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
})

C = {'sgd': '#1976D2', 'adam': '#F57C00', 'muon': '#7B1FA2',
     'shampoo': '#388E3C', 'ns': '#C62828', 'svd': '#546E7A'}

def polar_svd(G):
    U, _, Vt = np.linalg.svd(G, full_matrices=False)
    return U @ Vt

---
## 1. Newton-Schulz: быстрое полярное разложение

**Ключевая мысль:** Muon требует полярный фактор $UV^\top$ градиента. Полное SVD дорого и плохо ложится на GPU. Newton-Schulz использует **только матричные умножения**:

$$X_{k+1} = \frac{1}{2}X_k(3I - X_k^\top X_k), \qquad \varphi(\sigma) = \frac{3\sigma - \sigma^3}{2}$$

Неподвижная точка $\sigma=1$ — все сингулярные числа сходятся к единице (= ортогонализация).

In [ ]:
def newton_schulz_cubic(G, steps=15):
    """X_{k+1} = X(3I - X^T X) / 2.  Fixed point: σ=1."""
    X = G / np.linalg.norm(G, 'fro')
    n = X.shape[1]
    history = [X.copy()]
    for _ in range(steps):
        X = 0.5 * X @ (3 * np.eye(n) - X.T @ X)
        history.append(X.copy())
    return X, history

np.random.seed(42)
G = np.random.randn(128, 64)
P_exact = polar_svd(G)
_, history = newton_schulz_cubic(G, steps=15)

errors = [np.linalg.norm(h - P_exact, 'fro') / np.linalg.norm(P_exact, 'fro') for h in history]
sv_evo = [np.linalg.svd(h, compute_uv=False) for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.semilogy(errors, 'o-', color=C['ns'], ms=6, lw=2.5, zorder=5)
ax1.axhline(1e-10, color='gray', ls='--', alpha=0.5, label='$10^{-10}$')
ax1.set_xlabel('Итерация')
ax1.set_ylabel(r'$\|X_k - UV^\top\|_F / \|UV^\top\|_F$')
ax1.set_title('Квадратичная сходимость Newton-Schulz')
ax1.set_ylim(1e-16, 10)
ax1.legend(fontsize=11)

cmap = plt.cm.viridis(np.linspace(0.1, 0.9, 8))
for j in range(8):
    traj = [sv[j] for sv in sv_evo]
    label = f'$\\sigma_{{{j+1}}}$' if j in [0, 3, 7] else None
    ax2.plot(traj, color=cmap[j], lw=1.8, alpha=0.8, label=label)
ax2.axhline(1, color='k', ls='-', lw=1.5, alpha=0.3, label='Цель: $\\sigma=1$')
ax2.set_xlabel('Итерация')
ax2.set_ylabel('Сингулярное число')
ax2.set_title('Все $\\sigma_i \\to 1$: ортогонализация')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()
print(f'Ошибка на шаге 10: {errors[10]:.1e},  на шаге 13: {errors[13]:.1e}')

---
## 2. Спектр обновлений: SGD перекашивает, Muon выравнивает

**Ключевая мысль:** Градиенты нейросетей имеют сильно неравномерный спектр. SGD направляет почти всю «энергию» шага на первые сингулярные направления. Muon (полярный фактор) заменяет все сингулярные числа на 1 — **равномерный прогресс по всем направлениям**.

In [ ]:
np.random.seed(0)
m, n = 128, 64
U_t, _ = np.linalg.qr(np.random.randn(m, m))
V_t, _ = np.linalg.qr(np.random.randn(n, n))
sigmas = np.array([100 / (i+1)**1.5 for i in range(n)])
S = np.zeros((m, n)); np.fill_diagonal(S, sigmas)
G = U_t @ S @ V_t.T

sgd_upd = G / np.linalg.norm(G, 'fro')
sv_sgd = np.linalg.svd(sgd_upd, compute_uv=False)
sv_muon = np.linalg.svd(polar_svd(G), compute_uv=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
k = 30; x = np.arange(k)

ax1.semilogy(x, sv_sgd[:k], 's-', color=C['sgd'], ms=4, lw=2, label='SGD (∝ градиент)')
ax1.semilogy(x, sv_muon[:k], 'o-', color=C['muon'], ms=4, lw=2, label=r'Muon ($UV^\top$)')
ratio = sv_sgd[0]/sv_sgd[k-1]
ax1.annotate(f'{ratio:.0f}×', xy=(k-1, sv_sgd[k-1]), fontsize=11,
             fontweight='bold', color=C['sgd'],
             xytext=(k-12, sv_sgd[k-1]*0.12),
             arrowprops=dict(arrowstyle='->', color=C['sgd'], lw=1.5))
ax1.set_xlabel('Индекс'); ax1.set_ylabel('Сингулярное число')
ax1.set_title('Спектр обновлений'); ax1.legend(fontsize=11)

e_sgd = sv_sgd[:k]**2 / sv_sgd.sum()**2 * 100 * len(sv_sgd)
e_muon = sv_muon[:k]**2 / sv_muon.sum()**2 * 100 * len(sv_muon)
w = 0.35
ax2.bar(x-w/2, sv_sgd[:k]**2/sum(sv_sgd**2)*100, w, color=C['sgd'], alpha=.85, label='SGD')
ax2.bar(x+w/2, sv_muon[:k]**2/sum(sv_muon**2)*100, w, color=C['muon'], alpha=.85, label='Muon')
ax2.set_xlabel('Направление'); ax2.set_ylabel('Доля энергии, %')
ax2.set_title('Распределение энергии обновления'); ax2.legend(fontsize=11)

plt.tight_layout(); plt.show()
print(f'SGD  σ_max/σ_min = {sv_sgd[0]/sv_sgd[-1]:.0f}')
print(f'Muon σ_max/σ_min = {sv_muon[0]/sv_muon[-1]:.4f}')
print(f'SGD  top-3 = {sum(sv_sgd[:3]**2)/sum(sv_sgd**2)*100:.1f}% энергии')
print(f'Muon top-3 = {sum(sv_muon[:3]**2)/sum(sv_muon**2)*100:.1f}% энергии')

---
## 3. Инвариантность Muon к числу обусловленности

**Ключевая мысль:** На задаче $L(W) = \frac{1}{2}\text{tr}(W^\top A W B)$ с $A, B \succ 0$ и числом обусловленности $\kappa$, GD и Adam сильно деградируют при росте $\kappa$. Muon показывает **одинаковую скорость** при любом $\kappa$.

In [ ]:
def make_psd(dim, cond, seed):
    """PSD с eigenvalues в [1/cond, 1], ||A||=1."""
    rng = np.random.RandomState(seed)
    Q, _ = np.linalg.qr(rng.randn(dim, dim))
    return Q @ np.diag(np.geomspace(1.0/cond, 1.0, dim)) @ Q.T

def run_opt(W0, A, B, lr, steps, method):
    W = W0.copy()
    mb = vb = mub = np.zeros_like(W)
    losses = []
    for t in range(1, steps+1):
        losses.append(0.5 * np.trace(W.T @ A @ W @ B))
        G = A @ W @ B
        if method == 'gd':
            W -= lr * G
        elif method == 'adam':
            mb = .9*mb + .1*G; vb = .999*vb + .001*G**2
            W -= lr * (mb/(1-.9**t)) / (np.sqrt(vb/(1-.999**t)) + 1e-8)
        elif method == 'muon':
            mub = .95*mub + G
            U,_,Vt = np.linalg.svd(G + .95*mub, full_matrices=False)
            W -= lr * (U @ Vt)
    return np.array(losses)

m, n, steps = 32, 16, 500
conds = [1, 10, 100, 1000]
cmap_k = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(conds)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
cfgs = [('gd','GD',0.3), ('adam','Adam',0.01), ('muon','Muon',0.02)]

for ai, (meth, title, lr) in enumerate(cfgs):
    ax = axes[ai]
    for ci, kappa in enumerate(conds):
        A = make_psd(m, kappa, 10); B = make_psd(n, kappa, 20)
        np.random.seed(123)
        W0 = np.random.randn(m, n) * 0.1
        L = run_opt(W0, A, B, lr, steps, meth)
        ax.semilogy(L/L[0], lw=2, color=cmap_k[ci], label=f'κ={kappa:,}')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Итерация')
    if ai == 0: ax.set_ylabel(r'$L(W_k)/L(W_0)$')
    ax.legend(fontsize=9, loc='lower left')
    ax.set_ylim(1e-10, 10)

fig.suptitle('Зависимость сходимости от числа обусловленности κ', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

**Вывод:** GD катастрофически замедляется при росте κ (155 → 2 порядков). Adam деградирует мягче, но тоже (23 → 4 порядков). Muon стабилен: ~3 порядка при любом κ — ортогонализация «стирает» обусловленность.

---
## 4. Осцилляция Adam на $f(w) = w^2$

**Ключевая мысль:** На детерминированной $f(w)=w^2$ с достаточно большим lr Adam **осциллирует**, тратя сотни итераций на колебания. Причина: адаптивный знаменатель $\sqrt{\hat v}$ компенсирует уменьшение градиента — эффективный шаг остаётся почти постоянным.

In [ ]:
def run_1d(w0, lr, steps, method):
    w, mb, vb = w0, 0., 0.
    traj = [w]
    for t in range(1, steps+1):
        g = 2*w
        if method == 'sgd':
            w -= lr * g
        elif method == 'adam':
            mb = .9*mb + .1*g; vb = .999*vb + .001*g**2
            w -= lr * (mb/(1-.9**t)) / (np.sqrt(vb/(1-.999**t)) + 1e-8)
        elif method == 'norm':
            w -= lr * np.sign(g)
        traj.append(w)
    return np.array(traj)

N = 150
t_sgd  = run_1d(2., 0.3, N, 'sgd')
t_adam = run_1d(2., 1.0, N, 'adam')
t_norm = run_1d(2., 0.02, N, 'norm')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(t_sgd, color=C['sgd'], lw=2.5, label='SGD (lr=0.3)')
ax1.plot(t_adam, color=C['adam'], lw=2.5, label='Adam (lr=1.0)')
ax1.plot(t_norm, color=C['muon'], lw=2.5, label='NormGD ≈ Muon-1D (lr=0.02)')
ax1.axhline(0, color='k', lw=.8, alpha=.3)
ax1.set_xlabel('Итерация'); ax1.set_ylabel('$w$')
ax1.set_title('Траектории: Adam осциллирует')
ax1.legend(fontsize=10); ax1.set_ylim(-2, 3)

ax2.semilogy(t_sgd**2, color=C['sgd'], lw=2.5, label='SGD')
ax2.semilogy(t_adam**2, color=C['adam'], lw=2.5, label='Adam')
ax2.semilogy(t_norm**2, color=C['muon'], lw=2.5, label='NormGD')
ax2.set_xlabel('Итерация'); ax2.set_ylabel('$f(w) = w^2$')
ax2.set_title('SGD: экспоненциальная сходимость')
ax2.legend(fontsize=10)

plt.tight_layout(); plt.show()
print(f'SGD  на шаге 20:  f = {t_sgd[20]**2:.1e}')
print(f'Adam на шаге 20:  f = {t_adam[20]**2:.1e}  (ещё осциллирует!)')
print(f'Norm на шаге 100: f = {t_norm[100]**2:.1e}')

**Вывод:** SGD сходится экспоненциально (за ~20 шагов). Adam с lr=1 входит в затяжную осцилляцию на ~50 шагов, потеряв сотни итераций. NormGD (1D-аналог Muon: полярный фактор скаляра = sign) сходится линейно и стабильно — без осцилляций.

---
## 5. Shampoo(1 grad) ≡ Muon

**Ключевая мысль:** Shampoo с одним градиентом (без накопления): $(GG^\top)^{-1/4} G (G^\top G)^{-1/4} \equiv UV^\top$. Вся сложная система предобусловителей Кронекера сводится к одной операции — **полярному разложению**.

In [ ]:
def shampoo_1grad(G):
    def mpow(M, p):
        e, v = np.linalg.eigh(M)
        return v @ np.diag(np.maximum(e, 1e-12)**p) @ v.T
    return mpow(G@G.T, -.25) @ G @ mpow(G.T@G, -.25)

np.random.seed(7)
dims = [(32,16), (64,64), (16,48), (128,32)]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for i, (m, n) in enumerate(dims):
    G = np.random.randn(m, n) * (1+i)
    sv_G = np.linalg.svd(G, compute_uv=False)
    sv_sh = np.linalg.svd(shampoo_1grad(G), compute_uv=False)
    sv_mu = np.linalg.svd(polar_svd(G), compute_uv=False)
    diff = np.linalg.norm(shampoo_1grad(G) - polar_svd(G)) / np.linalg.norm(polar_svd(G))
    k = min(15, min(m,n)); x = np.arange(k); w=.25
    ax = axes[i]
    ax.bar(x-w, sv_G[:k]/sv_G[0], w, color=C['sgd'], alpha=.7, label='$G$')
    ax.bar(x, sv_sh[:k], w, color=C['shampoo'], alpha=.7, label='Shampoo')
    ax.bar(x+w, sv_mu[:k], w, color=C['muon'], alpha=.7, label='Muon')
    ax.axhline(1, color='k', ls='--', alpha=.3)
    ax.set_title(f'{m}×{n}\nΔ = {diff:.0e}', fontsize=10)
    ax.set_xlabel('σ index', fontsize=9)
    if i == 0: ax.legend(fontsize=7)

fig.suptitle('Shampoo(1 grad) = Muon: проверка для разных размерностей', fontsize=13)
plt.tight_layout(); plt.show()
print('Shampoo(1 grad) ≡ UV^T с машинной точностью ✓')

---
## 6. MLP на Fashion-MNIST: Muon vs Adam vs SGD

**Ключевая мысль:** На реальной задаче Muon (для 2D-параметров) + Adam (для bias/norm) ускоряет начальную сходимость MLP. Muon обновляет все сингулярные направления весовых матриц равномерно.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((.5,),(.5,))])
tr_loader = DataLoader(datasets.FashionMNIST('./data', True, download=True, transform=tf),
                       batch_size=256, shuffle=True)
te_loader = DataLoader(datasets.FashionMNIST('./data', False, download=True, transform=tf),
                       batch_size=1024)

In [ ]:
class MLP(nn.Module):
    def __init__(self, h=512):
        super().__init__()
        self.fc1, self.fc2, self.fc3 = nn.Linear(784,h), nn.Linear(h,h), nn.Linear(h,10)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x.flatten(1))))))

class MuonOpt(torch.optim.Optimizer):
    """Muon: Nesterov momentum + polar decomposition (SVD)."""
    def __init__(self, params, lr=.02, momentum=.95):
        super().__init__(params, dict(lr=lr, momentum=momentum))
    @torch.no_grad()
    def step(self):
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is None: continue
                s = self.state[p]
                if 'buf' not in s: s['buf'] = torch.zeros_like(p.grad)
                buf = s['buf']; mu = g['momentum']
                buf.mul_(mu).add_(p.grad)
                gn = p.grad + mu * buf
                if gn.dim() >= 2:
                    U,_,Vt = torch.linalg.svd(gn, full_matrices=False)
                    p.add_(U @ Vt, alpha=-g['lr'])
                else:
                    p.add_(gn, alpha=-g['lr'])

def epoch(model, loader, opt, train=True):
    model.train(train)
    tot_l, ok, n = 0., 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x); loss = F.cross_entropy(out, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        tot_l += loss.item()*x.size(0); ok += (out.argmax(1)==y).sum().item(); n += x.size(0)
    return tot_l/n, ok/n

In [ ]:
results = {}
for name in ['SGD+mom', 'Adam', 'Muon+Adam']:
    torch.manual_seed(42)
    model = MLP().to(device)
    if name == 'SGD+mom':
        opt = torch.optim.SGD(model.parameters(), lr=.01, momentum=.9)
    elif name == 'Adam':
        opt = torch.optim.Adam(model.parameters(), lr=.001)
    else:
        p2 = [p for p in model.parameters() if p.dim()>=2]
        p1 = [p for p in model.parameters() if p.dim()<2]
        o1, o2 = MuonOpt(p2, lr=.02), torch.optim.Adam(p1, lr=.001)
        class C:
            def __init__(s,a,b): s.a,s.b=a,b
            def zero_grad(s): s.a.zero_grad(); s.b.zero_grad()
            def step(s): s.a.step(); s.b.step()
        opt = C(o1,o2)
    trl, tel, tea = [],[],[]
    for e in range(25):
        l,_ = epoch(model, tr_loader, opt, True)
        _l,a = epoch(model, te_loader, opt, False)
        trl.append(l); tel.append(_l); tea.append(a)
        if (e+1)%5==0: print(f'{name:10s} ep {e+1:2d}: loss={l:.4f} acc={a:.4f}')
    results[name] = dict(trl=trl, tel=tel, tea=tea)
print('Done!')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
cm = {'SGD+mom': C['sgd'], 'Adam': C['adam'], 'Muon+Adam': C['muon']}
for nm, r in results.items():
    ax1.plot(r['trl'], lw=2.5, color=cm[nm], label=nm)
    ax2.plot(np.array(r['tea'])*100, lw=2.5, color=cm[nm], label=nm)
ax1.set_xlabel('Эпоха'); ax1.set_ylabel('Train Loss'); ax1.set_yscale('log')
ax1.set_title('Сходимость'); ax1.legend(fontsize=11)
ax2.set_xlabel('Эпоха'); ax2.set_ylabel('Test Accuracy, %')
ax2.set_title('Точность (Fashion-MNIST)'); ax2.legend(fontsize=11, loc='lower right')
plt.tight_layout(); plt.show()
for nm, r in results.items(): print(f'{nm:10s}: {r["tea"][-1]*100:.2f}%')

---
## Итоги

| # | Эксперимент | Ключевой вывод |
|:-:|:---|:---|
| 1 | Newton-Schulz | ~12 итераций matmul ≈ полярное разложение (SVD) |
| 2 | Спектр обновлений | SGD: top-3 забирают ~97% энергии. Muon: равномерно |
| 3 | Число обусловленности | GD/Adam деградируют при κ↑. Muon **инвариантен** |
| 4 | $f(w) = w^2$ | Adam осциллирует ~50 итераций. SGD и Muon-1D — нет |
| 5 | Shampoo ≡ Muon | $(GG^\top)^{-1/4}G(G^\top G)^{-1/4} = UV^\top$ |
| 6 | Fashion-MNIST | Muon+Adam ускоряет начальную сходимость MLP |